# AutoFill API — Python Quickstart

Subscribe on [RapidAPI](https://rapidapi.com/12devs-12devs-default/api/autofill), set `.env`, then **Run All**.

Live demo: [autofill.12devs.info](https://autofill.12devs.info/) · React example: [react-tryit](../react-tryit/)

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

from autofill_client import AutofillClient, fields_to_dataframe

load_dotenv()

def require_env(name: str) -> str:
    v = os.getenv(name, "").strip()
    if not v:
        raise RuntimeError(f"Set {name} in .env (see .env.example)")
    return v

client = AutofillClient(
    api_key=require_env("RAPIDAPI_KEY"),
    api_host=require_env("RAPIDAPI_HOST"),
    base_url=require_env("RAPIDAPI_BASE_URL"),
)
SAMPLE = Path("samples/sample-invoice.pdf")
print("Client ready.", SAMPLE.exists() and "Sample PDF found" or "Sample PDF missing")

In [ ]:
# Optional health check
print(client.ping())

In [ ]:
%%time
uploaded = client.upload(SAMPLE)
auto_result = client.recognize(uploaded["filePath"], mode="auto")
print(json.dumps(auto_result.get("fields", auto_result), indent=2, ensure_ascii=False)[:4000])

In [ ]:
%%time
import json as _json

templates = _json.loads(Path("data/templates.json").read_text(encoding="utf-8"))
invoice_fields = next(t["fields"] for t in templates if t["presetKey"] == "commercial_invoice")

manual_result = client.recognize(
    uploaded["filePath"],
    mode="manual",
    fields=invoice_fields,
)
print(f"Sent {len(invoice_fields)} fields (Commercial Invoice preset)")

In [ ]:
df = fields_to_dataframe(manual_result)
df

## Troubleshooting

- **401** — check `RAPIDAPI_KEY` and `RAPIDAPI_HOST` match the Hub Endpoints tab.
- **Timeout** — recognize may take up to **120 s** on large PDFs.
- **30 MB** upload limit on the gateway.
- **Empty OCR / fields** — try another preset or `mode: auto`.

In [ ]:
meta = manual_result.get("meta") or auto_result.get("meta")
if meta:
    print(json.dumps(meta, indent=2))